# Tracing and Observability

Every run in this series was already traced for you. A **trace** is a diary of everything an agent did, and a **span**
is one line in that diary (a model call, a tool call, a handoff). When an agent does something strange in production,
the trace is how you find out why.

This episode shows two things: how to write your own **logger** (a `TracingProcessor`) that notes down every step, and
how to hide private data so secrets never leave your app.


## Write a logger with `TracingProcessor`

Each time a step finishes, the SDK calls `on_span_end` on your logger. That is the only method we use: it writes one
line for the step. For a tool call it also keeps what went **in** and what came **out**, so we can see them later.
`set_trace_processors([...])` tells the SDK to send every step to our logger.


In [1]:
from agents import (
    TracingProcessor,  # the base class for your own place to send traces
)


# Our simple logger, registered with set_trace_processors(); the SDK calls on_span_end for every finished step
class SimpleLogger(TracingProcessor):
    # Makes the empty notebook where on_span_end writes
    def __init__(self) -> None:
        self.lines: list[str] = []  # one line per finished step
        self.tool_input: str | None = None  # what went into the tool
        self.tool_output: str | None = None  # what came out of the tool

    # The one method that matters: runs when a step finishes, and writes it down
    def on_span_end(self, span: object) -> None:
        # span_data.type is the kind of step: "function" (tool), "agent", "turn", ...
        self.lines.append(span.span_data.type)
        # only tool steps have an input and an output
        if span.span_data.type == "function":
            self.tool_input = span.span_data.input
            self.tool_output = span.span_data.output

    # Required by the interface, not used here: runs when a trace opens
    def on_trace_start(self, trace: object) -> None:
        pass

    # Required by the interface, not used here: runs when a trace closes
    def on_trace_end(self, trace: object) -> None:
        pass

    # Required by the interface, not used here: runs when a step opens
    def on_span_start(self, span: object) -> None:
        pass

    # Required by the interface, not used here: a real logger would send waiting lines
    def force_flush(self) -> None:
        pass

    # Required by the interface, not used here: a real logger would close its connection
    def shutdown(self) -> None:
        pass

## Give the agent a secret tool

This tool returns something classified, so it is a good one to hide from the logs.


In [2]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: a "classified" lookup whose input and output we hide in the demo below
@function_tool
def get_ninja_rank(name: str) -> str:
    """Look up a ninja's rank (classified, so it is a good one to hide)."""
    return "Kage-level (classified)"

## Hide private data with `trace_include_sensitive_data`

Traces can hold real tool inputs and outputs. `RunConfig(trace_include_sensitive_data=False)` blanks them before they
are sent anywhere. The demo runs the same tool call twice, with private data on and then off, and prints what our
logger wrote.


In [3]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
    set_trace_processors,  # swaps out the SDK's default trace processors
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
    function_call,  # makes one scripted tool-call step for ScriptedModel
)


# Demo: run the same tool call with private data on, then off, and print what the logger wrote
async def demo_tracing() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="test",
        tools=[get_ninja_rank],
    )
    for include_sensitive in (True, False):
        # Initialize the tracing processor
        logger = SimpleLogger()
        set_trace_processors([logger])
        # Script the model
        model = ScriptedModel(
            [
                [function_call("get_ninja_rank", {"name": "Naruto Uzumaki"}, call_id="c1")],
                [assistant_message("Kage-level.")],
            ]
        )
        # Run the Agent
        await Runner.run(
            agent,
            "who is it",
            run_config=RunConfig(
                model=model,
                # trace_include_sensitive_data: False blanks tool input and output before they are sent out
                trace_include_sensitive_data=include_sensitive,
            ),
        )
        # Inspect the result
        # proves our logger saw the tool step (the "function" line) on every run
        print(f"[steps, sensitive={include_sensitive}]", logger.lines)
        # proves True keeps the real tool input and output, and False blanks both
        print(f"[tool input, sensitive={include_sensitive}]", logger.tool_input)
        print(f"[tool output, sensitive={include_sensitive}]", logger.tool_output)


await demo_tracing()
# Outside a notebook: asyncio.run(demo_tracing())

[steps, sensitive=True] ['function', 'turn', 'turn', 'agent', 'task']
[tool input, sensitive=True] {"name":"Naruto Uzumaki"}
[tool output, sensitive=True] Kage-level (classified)
[steps, sensitive=False] ['function', 'turn', 'turn', 'agent', 'task']
[tool input, sensitive=False] None
[tool output, sensitive=False] None


## Summary

- A `TracingProcessor` is your own logger: the SDK calls `on_span_end` for every finished step, and you decide what to
  write down. `set_trace_processors` plugs it in.
- `RunConfig(trace_include_sensitive_data=False)` blanks tool input and output before they leave your app, so private
  data stays out of your traces.
